# UMUD EXP76 controlled diagnostic segmentation matrix

This is the notebook I want run tonight.

It is deliberately controlled, not flashy: one segmentation axis changes at a time, while the measurement side stays on the public-safe stack. It avoids the rejected geometry/scale proxies and writes enough evidence that a closed tab or failed run still leaves logs, configs, summaries, submissions, and debug masks.

Setup:
1. Add Input -> **UMUD Challenge: Muscle Architecture in Ultrasound Data**.
2. Accelerator -> GPU.
3. Internet -> On.
4. Run All.

Expected runtime: likely several hours on a T4 for all six runs. If you want to shorten it, edit only `MAX_RUNS` in the run cell.

Audit framing:
- External ultrasound sources support controlled segmentation plus explicit measurement geometry.
- Antagonistic sources warn that heavy augmentation, pseudo-labels, and broad postprocess changes can hurt.
- This notebook therefore runs controlled supervised variants first. Masked pretraining and the classical line extractor are real next ideas, but they require new implementation code and are not pretended here.


In [ ]:
# Fast data preflight. This fails before install/training if the competition input is not attached.
from pathlib import Path
import os

IMG_EXTS = ('.tif', '.tiff', '.png', '.jpg', '.jpeg', '.bmp')
LEAVES = {
    'apo_img': ['apo_images_new_model_v1'],
    'apo_msk': ['apo_masks_new_model_v1'],
    'fasc_img': ['fasc_images_new_model_v1'],
    'fasc_msk': ['fasc_masks_new_model_v1'],
    'test': ['test_set_v2', 'test_images_v2'],
}
TERMINAL_LEAVES = {'apo_images_new_model_v1', 'apo_masks_new_model_v1', 'fasc_images_new_model_v1', 'fasc_masks_new_model_v1', 'test_set_v2'}

def list_images(d):
    try:
        return [p for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]
    except OSError:
        return []

def print_tree(root, max_depth=3, depth=0, budget=None):
    budget = [300] if budget is None else budget
    try:
        entries = sorted(root.iterdir())
    except OSError:
        return
    for p in entries:
        if budget[0] <= 0:
            return
        budget[0] -= 1
        print('  ' * depth + ('[D] ' if p.is_dir() else '    ') + p.name)
        if p.is_dir() and depth < max_depth:
            print_tree(p, max_depth, depth + 1, budget)

def index_root(root):
    index = {}
    for dirpath, dirnames, _files in os.walk(root):
        for dn in dirnames:
            index.setdefault(dn, []).append(Path(dirpath) / dn)
        dirnames[:] = [d for d in dirnames if d not in TERMINAL_LEAVES]
    return index

def resolve_key(index, key):
    for leaf in LEAVES[key]:
        cands = index.get(leaf, [])
        if key == 'test':
            cands = sorted(cands, key=lambda d: len(list_images(d)), reverse=True)
            cands = [d for d in cands if list_images(d)]
        if cands:
            return cands[0]
    return None

roots = sorted(p for p in Path('/kaggle/input').iterdir() if p.is_dir()) if Path('/kaggle/input').exists() else []
print('Mounted roots:')
for root in roots:
    print(' -', root)

resolved = None
best_n = -1
for root in roots:
    idx = index_root(root)
    dirs = {key: resolve_key(idx, key) for key in LEAVES}
    n = sum(v is not None for v in dirs.values())
    if n > best_n:
        resolved = dirs
        best_n = n
resolved = resolved or {key: None for key in LEAVES}

print('\nResolved data folders:')
for key, path in resolved.items():
    count = len(list_images(path)) if path else 0
    print(f'  {key:8s} -> {path} ({count} images)' if path else f'  {key:8s} -> None')
missing = [key for key, path in resolved.items() if path is None]
if missing:
    print('\nMounted tree for debugging:')
    for root in roots or [Path('/kaggle/input')]:
        print(f'[root] {root}')
        print_tree(root)
    raise RuntimeError(f'Missing UMUD folders: {missing}. Attach the competition input, then Run All again.')

assert len(list_images(resolved['test'])) == 309, f"expected 309 test images, found {len(list_images(resolved['test']))}"
print('\nPreflight OK: competition data is mounted and test set has 309 images.')


In [ ]:
# Install dependencies and download current repo scripts. Internet must be On.
import subprocess
import sys
import urllib.request
from pathlib import Path

WORK = Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch', 'albumentations'])

RAW_BASE = 'https://raw.githubusercontent.com/LacrimaeAware/kaggle-fun/main/umud-muscle-architecture'
SCRIPTS = [
    'segment_then_measure.py',
    'tick_calibration.py',
    'scale_ticks.py',
    'subpixel_spacing.py',
    'scale_ocr.py',
]
for script in SCRIPTS:
    urllib.request.urlretrieve(f'{RAW_BASE}/{script}', str(WORK / script))
    assert (WORK / script).exists(), f'{script} failed to download; check Internet = On'

EXPECTED_VERSION = '2026-06-13.03'
got = 'MISSING'
for line in (WORK / 'segment_then_measure.py').read_text(encoding='utf-8').splitlines():
    if line.startswith('PIPELINE_VERSION'):
        got = line.split('"')[1]
        break
print(f'>>> downloaded pipeline VERSION: {got}   (expected {EXPECTED_VERSION}) <<<')
assert got == EXPECTED_VERSION, 'STALE script: GitHub raw served an old file. Wait 1-2 minutes and re-run this cell.'

import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')


In [ ]:
# Run EXP76 controlled diagnostic matrix.
import json
import os
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

WORK = Path('/kaggle/working')
os.chdir(WORK)
LOG_DIR = WORK / 'seg76_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
STATUS_PATH = WORK / 'seg76_controlled_status.json'
SUMMARY_PATH = WORK / 'seg76_controlled_summary.csv'
MANIFEST_PATH = WORK / 'seg76_run_manifest.csv'

COMMON_ENV = {
    'PYTHONUNBUFFERED': '1',
    # Public-safe measurement side: keep accepted pieces, avoid rejected broad proxies.
    'UMUD_SCALE_ROUTER': '1',
    'UMUD_TTA': '1',
    'UMUD_TEMPORAL_SMOOTH': '1',
    'UMUD_USE_CALIBRATED_MT': '1',
    'UMUD_USE_CALIBRATED_FL': '0',
    'UMUD_FRAGMENT_FL': '1',
    'UMUD_USE_IDENTITY_FL': '1',
    'UMUD_FL_RECENTER': '1',
    'UMUD_FL_IDENTITY_BLEND': '0',
    'UMUD_TOP_BOUNDARY_MODE': 'line',
    'UMUD_MT_MODE': 'perp_center',
    # Controlled diagnostics.
    'UMUD_AUTO_THRESHOLD': '1',
    'UMUD_THRESHOLD_SWEEP': '0.20,0.28,0.36,0.44,0.52,0.60,0.68',
    'UMUD_APO_TARGET_MODE': 'binary',
    'UMUD_APO_MASK_THRESHOLD': '0.5',
    'UMUD_FASC_POSTPROCESS': 'threshold',
    'UMUD_FASC_MIN_ANG': '6',
    'UMUD_SAVE_PRED_DEBUG': '40',
}

RUNS = [
    {
        'run_id': 'seg76_00_control_binary_thr',
        'reason': 'control: 512 U-Net, binary masks, auto threshold, public-safe measurement stack',
        'submit_priority': 'submit only if it beats prior segmentation candidates or if all others fail and QC is sane',
        'env': {
            'UMUD_EPOCHS': '24',
            'UMUD_IMG_SIZE': '512',
            'UMUD_TTA_EXTRA_SIZE': '640',
            'UMUD_BATCH_SIZE': '6',
            'UMUD_MODEL_ARCH': 'unet',
            'UMUD_MODEL_ENCODER': 'resnet34',
            'UMUD_MODEL_ENCODER_WEIGHTS': 'imagenet',
            'UMUD_LOSS_MODE': 'dice_bce',
            'UMUD_AUG_LEVEL': 'light',
            'UMUD_CLAHE': '0',
            'UMUD_FASC_TARGET_MODE': 'binary',
            'UMUD_FASC_MIN_AREA': '40',
            'UMUD_FASC_POS_WEIGHT': '0',
            'UMUD_WEIGHTS_TAG': 'seg76_00',
        },
    },
    {
        'run_id': 'seg76_01_soft5_thr',
        'reason': 'one-axis target test: soft fascicle target, no skeleton postprocess, same architecture/loss',
        'submit_priority': 'good if fasc Dice and fragment counts improve without flooding masks',
        'env': {
            'UMUD_EPOCHS': '24',
            'UMUD_IMG_SIZE': '512',
            'UMUD_TTA_EXTRA_SIZE': '640',
            'UMUD_BATCH_SIZE': '6',
            'UMUD_MODEL_ARCH': 'unet',
            'UMUD_MODEL_ENCODER': 'resnet34',
            'UMUD_MODEL_ENCODER_WEIGHTS': 'imagenet',
            'UMUD_LOSS_MODE': 'dice_bce',
            'UMUD_AUG_LEVEL': 'light',
            'UMUD_CLAHE': '0',
            'UMUD_FASC_TARGET_MODE': 'soft5',
            'UMUD_FASC_MIN_AREA': '24',
            'UMUD_FASC_POS_WEIGHT': '0',
            'UMUD_WEIGHTS_TAG': 'seg76_01',
        },
    },
    {
        'run_id': 'seg76_02_dilate3_thr',
        'reason': 'one-axis target test: dilated fascicle target to give thin structures more support',
        'submit_priority': 'good if usable fragment count rises without PA/FL distribution drift',
        'env': {
            'UMUD_EPOCHS': '24',
            'UMUD_IMG_SIZE': '512',
            'UMUD_TTA_EXTRA_SIZE': '640',
            'UMUD_BATCH_SIZE': '6',
            'UMUD_MODEL_ARCH': 'unet',
            'UMUD_MODEL_ENCODER': 'resnet34',
            'UMUD_MODEL_ENCODER_WEIGHTS': 'imagenet',
            'UMUD_LOSS_MODE': 'dice_bce',
            'UMUD_AUG_LEVEL': 'light',
            'UMUD_CLAHE': '0',
            'UMUD_FASC_TARGET_MODE': 'dilate3',
            'UMUD_FASC_MIN_AREA': '24',
            'UMUD_FASC_POS_WEIGHT': '0',
            'UMUD_WEIGHTS_TAG': 'seg76_02',
        },
    },
    {
        'run_id': 'seg76_03_tversky_binary_thr',
        'reason': 'one-axis loss test: Tversky recall pressure with binary target, no target-shape confound',
        'submit_priority': 'good if fasc recall improves without noisy component explosion',
        'env': {
            'UMUD_EPOCHS': '24',
            'UMUD_IMG_SIZE': '512',
            'UMUD_TTA_EXTRA_SIZE': '640',
            'UMUD_BATCH_SIZE': '6',
            'UMUD_MODEL_ARCH': 'unet',
            'UMUD_MODEL_ENCODER': 'resnet34',
            'UMUD_MODEL_ENCODER_WEIGHTS': 'imagenet',
            'UMUD_LOSS_MODE': 'dice_tversky',
            'UMUD_AUG_LEVEL': 'light',
            'UMUD_CLAHE': '0',
            'UMUD_FASC_TARGET_MODE': 'binary',
            'UMUD_FASC_MIN_AREA': '40',
            'UMUD_FASC_POS_WEIGHT': '0',
            'UMUD_WEIGHTS_TAG': 'seg76_03',
        },
    },
    {
        'run_id': 'seg76_04_unetpp_binary_thr',
        'reason': 'one-axis architecture test: U-Net++ at 512 with binary target/loss unchanged',
        'submit_priority': 'good if architecture improves thin detail without degrading apo',
        'env': {
            'UMUD_EPOCHS': '24',
            'UMUD_IMG_SIZE': '512',
            'UMUD_TTA_EXTRA_SIZE': '640',
            'UMUD_BATCH_SIZE': '4',
            'UMUD_MODEL_ARCH': 'unetplusplus',
            'UMUD_MODEL_ENCODER': 'resnet34',
            'UMUD_MODEL_ENCODER_WEIGHTS': 'imagenet',
            'UMUD_LOSS_MODE': 'dice_bce',
            'UMUD_AUG_LEVEL': 'light',
            'UMUD_CLAHE': '0',
            'UMUD_FASC_TARGET_MODE': 'binary',
            'UMUD_FASC_MIN_AREA': '40',
            'UMUD_FASC_POS_WEIGHT': '0',
            'UMUD_WEIGHTS_TAG': 'seg76_04',
        },
    },
    {
        'run_id': 'seg76_05_soft5_tversky_thr',
        'reason': 'optional combo: soft fascicle target plus Tversky after one-axis checks',
        'submit_priority': 'submit only if it clearly beats the control diagnostics',
        'env': {
            'UMUD_EPOCHS': '24',
            'UMUD_IMG_SIZE': '512',
            'UMUD_TTA_EXTRA_SIZE': '640',
            'UMUD_BATCH_SIZE': '6',
            'UMUD_MODEL_ARCH': 'unet',
            'UMUD_MODEL_ENCODER': 'resnet34',
            'UMUD_MODEL_ENCODER_WEIGHTS': 'imagenet',
            'UMUD_LOSS_MODE': 'dice_tversky',
            'UMUD_AUG_LEVEL': 'light',
            'UMUD_CLAHE': '0',
            'UMUD_FASC_TARGET_MODE': 'soft5',
            'UMUD_FASC_MIN_AREA': '24',
            'UMUD_FASC_POS_WEIGHT': '0',
            'UMUD_WEIGHTS_TAG': 'seg76_05',
        },
    },
]

MAX_RUNS = 6
MAX_WALL_HOURS = 11.5
FORCE_RERUN = False

manifest_rows = []
for run in RUNS[:MAX_RUNS]:
    row = {'run_id': run['run_id'], 'reason': run['reason'], 'submit_priority': run['submit_priority']}
    row.update(run['env'])
    manifest_rows.append(row)
pd.DataFrame(manifest_rows).to_csv(MANIFEST_PATH, index=False)
print('Run manifest:')
print(pd.DataFrame(manifest_rows)[['run_id', 'reason', 'submit_priority', 'UMUD_MODEL_ARCH', 'UMUD_LOSS_MODE', 'UMUD_FASC_TARGET_MODE']].to_string(index=False))


def now_iso():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')


def load_summaries():
    if SUMMARY_PATH.exists():
        try:
            return pd.read_csv(SUMMARY_PATH).to_dict('records')
        except Exception:
            return []
    return []


def write_status(payload):
    payload = dict(payload)
    payload['updated_at_utc'] = now_iso()
    STATUS_PATH.write_text(json.dumps(payload, indent=2, default=str), encoding='utf-8')


def append_or_replace_summary(rows, row):
    rows = [r for r in rows if r.get('run_id') != row.get('run_id')]
    rows.append(row)
    pd.DataFrame(rows).to_csv(SUMMARY_PATH, index=False)
    return rows


def valid_submission(path):
    if not path.exists():
        return False
    try:
        sub = pd.read_csv(path)
    except Exception:
        return False
    return sub.shape == (309, 4) and sub['image_id'].is_unique and not sub[['pa_deg', 'fl_mm', 'mt_mm']].isna().any().any()


def collect_outputs(run_id):
    produced = []
    for src_name, dst_name in [
        ('submission_segmentation.csv', f'submission_{run_id}.csv'),
        ('calibration_measurement_debug.csv', f'calibration_measurement_debug_{run_id}.csv'),
    ]:
        src = WORK / src_name
        if src.exists():
            dst = WORK / dst_name
            shutil.copy2(src, dst)
            produced.append(dst.name)
    return produced


def parse_log_metrics(log_path):
    metrics = {}
    if not log_path.exists():
        return metrics
    txt = log_path.read_text(encoding='utf-8', errors='replace')
    for target, val in re.findall(r'\[(apo|fasc)\] best dice ([0-9.]+)', txt):
        metrics[f'{target}_best_dice'] = float(val)
    for target, val in re.findall(r'\[(apo|fasc)\] selected inference threshold ([0-9.]+)', txt):
        metrics[f'{target}_selected_thr'] = float(val)
    return metrics


def summarize_outputs(run_id, env=None, elapsed_min=None, status='ok', error='', produced=None, log_path=None):
    stats = {
        'run_id': run_id,
        'status': status,
        'elapsed_min': elapsed_min,
        'error': error,
        'produced': json.dumps(produced or []),
        'log_path': str(log_path) if log_path else '',
    }
    env = env or {}
    for key in ['UMUD_MODEL_ARCH', 'UMUD_LOSS_MODE', 'UMUD_AUG_LEVEL', 'UMUD_FASC_TARGET_MODE', 'UMUD_FASC_POSTPROCESS', 'UMUD_IMG_SIZE', 'UMUD_EPOCHS']:
        stats[key] = env.get(key, '')
    if log_path:
        stats.update(parse_log_metrics(log_path))
    sub_path = WORK / f'submission_{run_id}.csv'
    dbg_path = WORK / f'calibration_measurement_debug_{run_id}.csv'
    if sub_path.exists():
        sub = pd.read_csv(sub_path)
        stats.update({
            'rows': int(len(sub)),
            'pa_mean': float(sub['pa_deg'].mean()),
            'pa_std': float(sub['pa_deg'].std()),
            'pa_min': float(sub['pa_deg'].min()),
            'pa_max': float(sub['pa_deg'].max()),
            'fl_mean': float(sub['fl_mm'].mean()),
            'fl_std': float(sub['fl_mm'].std()),
            'fl_min': float(sub['fl_mm'].min()),
            'fl_max': float(sub['fl_mm'].max()),
            'mt_mean': float(sub['mt_mm'].mean()),
            'mt_std': float(sub['mt_mm'].std()),
            'mt_min': float(sub['mt_mm'].min()),
            'mt_max': float(sub['mt_mm'].max()),
            'pa_clip_count': int(((sub['pa_deg'] <= 5.001) | (sub['pa_deg'] >= 44.999)).sum()),
            'fl_clip_count': int(((sub['fl_mm'] <= 20.001) | (sub['fl_mm'] >= 179.999)).sum()),
            'mt_clip_count': int(((sub['mt_mm'] <= 2.001) | (sub['mt_mm'] >= 49.999)).sum()),
        })
    if dbg_path.exists():
        dbg = pd.read_csv(dbg_path)
        if 'calibration_method' in dbg:
            stats['scaled_images'] = int((dbg['calibration_method'] != 'none').sum())
            stats['calibration_methods'] = json.dumps(dbg['calibration_method'].value_counts(dropna=False).head(12).to_dict())
        for col in ['fl_fragment_n', 'fl_fragment_median_px', 'fl_px', 'mt_px']:
            if col in dbg:
                vals = pd.to_numeric(dbg[col], errors='coerce')
                stats[f'{col}_mean'] = float(vals.mean()) if vals.notna().any() else None
                stats[f'{col}_non_null'] = int(vals.notna().sum())
    return stats


summaries = load_summaries()
matrix_start = time.time()
write_status({'state': 'matrix_start', 'max_runs': MAX_RUNS, 'max_wall_hours': MAX_WALL_HOURS, 'force_rerun': FORCE_RERUN, 'manifest_path': str(MANIFEST_PATH)})

for run in RUNS[:MAX_RUNS]:
    if (time.time() - matrix_start) / 3600.0 > MAX_WALL_HOURS:
        msg = f'stopping before {run["run_id"]}: wall budget reached'
        print(msg)
        write_status({'state': 'wall_budget_stop', 'message': msg, 'summary_path': str(SUMMARY_PATH)})
        break

    run_id = run['run_id']
    log_path = LOG_DIR / f'{run_id}.log'
    final_sub = WORK / f'submission_{run_id}.csv'
    if valid_submission(final_sub) and not FORCE_RERUN:
        print('\nSKIP completed', run_id, '->', final_sub.name)
        row = summarize_outputs(run_id, env={**COMMON_ENV, **run['env']}, status='skipped_completed', produced=[final_sub.name], log_path=log_path)
        summaries = append_or_replace_summary(summaries, row)
        continue

    print('\n' + '=' * 100)
    print('START', run_id)
    print(run['reason'])
    print('priority:', run['submit_priority'])
    print('log:', log_path)
    print('=' * 100)

    env = os.environ.copy()
    env.update(COMMON_ENV)
    env.update(run['env'])
    config = {
        'run_id': run_id,
        'reason': run['reason'],
        'submit_priority': run['submit_priority'],
        'common_env': COMMON_ENV,
        'run_env': run['env'],
        'merged_env': {k: env[k] for k in sorted(set(COMMON_ENV) | set(run['env']))},
    }
    (WORK / f'run_config_{run_id}.json').write_text(json.dumps(config, indent=2), encoding='utf-8')

    start = time.time()
    write_status({'state': 'running', 'run_id': run_id, 'started_at_utc': now_iso(), 'log_path': str(log_path), 'config': config})
    summaries = append_or_replace_summary(summaries, {'run_id': run_id, 'status': 'running', 'elapsed_min': 0.0, 'error': '', 'log_path': str(log_path)})

    with log_path.open('w', encoding='utf-8', errors='replace') as log:
        log.write(f'START {run_id} {now_iso()}\n')
        log.write(json.dumps(config, indent=2) + '\n')
        log.flush()
        proc = subprocess.Popen([sys.executable, '-u', 'segment_then_measure.py'], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        returncode = proc.wait()
        log.write(f'END {run_id} {now_iso()} returncode={returncode}\n')

    elapsed_min = (time.time() - start) / 60.0
    produced = collect_outputs(run_id)
    if returncode == 0 and valid_submission(final_sub):
        row = summarize_outputs(run_id, env={**COMMON_ENV, **run['env']}, elapsed_min=elapsed_min, status='ok', error='', produced=produced, log_path=log_path)
        print('FINISHED', run_id, f'in {elapsed_min:.1f} min')
    else:
        row = summarize_outputs(run_id, env={**COMMON_ENV, **run['env']}, elapsed_min=elapsed_min, status='failed', error=f'returncode={returncode}; produced={produced}', produced=produced, log_path=log_path)
        print('FAILED', run_id, row['error'])

    summaries = append_or_replace_summary(summaries, row)
    write_status({'state': row['status'], 'run_id': run_id, 'elapsed_min': elapsed_min, 'log_path': str(log_path), 'summary_path': str(SUMMARY_PATH)})
    print('Summary so far:')
    cols = [c for c in ['run_id','status','elapsed_min','apo_best_dice','fasc_best_dice','apo_selected_thr','fasc_selected_thr','pa_mean','fl_mean','mt_mean','fl_fragment_n_mean','scaled_images','error'] if c in pd.DataFrame(summaries).columns]
    print(pd.DataFrame(summaries)[cols].to_string(index=False))

write_status({'state': 'matrix_complete', 'summary_path': str(SUMMARY_PATH), 'runs': summaries})
print('\nEXP76 CONTROLLED MATRIX COMPLETE OR RECORDED.')
print('Summary file:', SUMMARY_PATH)
print('Status file:', STATUS_PATH)
print('Manifest file:', MANIFEST_PATH)
print('Logs:', LOG_DIR)


In [ ]:
# Bundle submissions, debug files, configs, summaries, logs/status, debug masks, and weights into one zip.
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display
import os

WORK = Path('/kaggle/working')
os.chdir(WORK)
zip_path = WORK / 'umud_seg76_controlled_diagnostics_outputs.zip'
patterns = [
    'submission_*.csv',
    'calibration_measurement_debug_*.csv',
    'seg76_controlled_summary.csv',
    'seg76_controlled_status.json',
    'seg76_run_manifest.csv',
    'run_config_*.json',
    'seg_*.pt',
    'seg76_logs/*.log',
    'pred_debug_*/*.png',
]
include_paths = []
for pattern in patterns:
    include_paths.extend(sorted(WORK.glob(pattern)))
include_paths = sorted({p.resolve(): p for p in include_paths if p.exists()}.values(), key=lambda p: str(p))

with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as zf:
    for path in include_paths:
        zf.write(path, arcname=str(path.relative_to(WORK)))

print('Created:', zip_path.name, f'({zip_path.stat().st_size:,} bytes)')
print('Contents:')
for path in include_paths:
    print(' -', path.relative_to(WORK), f'({path.stat().st_size:,} bytes)')

print('\nDownload links:')
display(FileLink(zip_path.name))
for path in include_paths:
    rel = path.relative_to(WORK)
    if path.name.startswith('submission_') or path.name in {'seg76_controlled_summary.csv', 'seg76_controlled_status.json', 'seg76_run_manifest.csv'}:
        display(FileLink(str(rel)))
